# PlacePulse AI: Technical Resume Domain Classification & NLP Engine
## Viva & Publication Ready Lab Notebook (03_resume_nlp_classification.ipynb)
**Author:** PlacePulse AI Engineering Team  
**Dataset:** Gaurav Dutta Updated Resume Dataset (962 records, 25+ technical categories)  
**NLP Architecture:** TF-IDF Sublinear Vectorization + LinearSVC / Multinomial Naive Bayes + LinkedIn Skill Gap Extraction.


In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("NLP Pipeline packages loaded!")


### 1. Ingest Resume Corpus & Category Distribution


In [ ]:
resume_path = os.path.join("..", "data", "raw", "UpdatedResumeDataSet.csv")
df_res = pd.read_csv(resume_path)
print(f"Total Resumes: {len(df_res)} across {df_res['Category'].nunique()} roles")

plt.figure(figsize=(12, 6))
top_roles = df_res["Category"].value_counts()
sns.barplot(x=top_roles.values, y=top_roles.index, palette="mako")
plt.title("Resume Corpus Domain Distribution", fontsize=14, fontweight="bold")
plt.xlabel("Sample Count")
plt.ylabel("Tech Job Role")
plt.tight_layout()
plt.show()


### 2. Inspect Benchmark Metrics for LinearSVC vs Naive Bayes


In [ ]:
nlp_benchmarks_path = os.path.join("..", "models", "nlp_benchmarks.json")
with open(nlp_benchmarks_path, "r") as f:
    nlp_benchmarks = json.load(f)

print(f"Active NLP Classifier: {nlp_benchmarks['production_model']}")
pd.DataFrame(nlp_benchmarks["leaderboard"]).T[["test_accuracy", "macro_f1", "weighted_f1"]]


### 3. Live Inference Demo & Skill-Gap Extraction
Demonstrating real-time text screening and missing prerequisite detection.


In [ ]:
import sys
sys.path.append(os.path.join("..", "src"))
from feature_engineering import analyze_skill_gaps, clean_resume_text

sample_cv = """
Full Stack Software Developer with 2+ years experience in Java, Spring Boot, Microservices,
RESTful APIs, PostgreSQL, Docker, Git and Maven. Built high throughput backend services.
"""

vectorizer = joblib.load(os.path.join("..", "models", "resume_tfidf.joblib"))
clf = joblib.load(os.path.join("..", "models", "resume_classifier.joblib"))
encoder = joblib.load(os.path.join("..", "models", "resume_label_encoder.joblib"))

cleaned = clean_resume_text(sample_cv)
vec = vectorizer.transform([cleaned])
pred_role = encoder.inverse_transform(clf.predict(vec))[0]

gaps = analyze_skill_gaps(sample_cv, pred_role)

print(f"Predicted Role: {pred_role}")
print(f"Competency Match: {gaps['match_percentage']}%")
print(f"Matched Skills: {gaps['matched_skills']}")
print(f"Missing Prerequisite Skills: {gaps['missing_skills']}")
